# Preprocesamiento: detección automática por fotograma con YOLO26

Ejecuta **YOLO26** preentrenado (COCO) sobre cada fotograma de la fase `frames` y guarda, por fotograma:

- **Conteos** por clase: `n_person`, `n_car`, `n_bus`, `n_truck`, `n_motorcycle` y `n_vehiculos` (suma de vehículos).
- **Semáforos por color**: `n_traffic_light_red/yellow/green/unknown` y, con una heurística de tamaño y posición, `n_pedestrian_light_red/green`.
- `pedestrian_group`: al menos `min_personas_grupo` personas en la imagen.

El código de inferencia y color está en `src/deteccion.py` (compartido con la pre-anotación de CVAT).

**Uso:** estos conteos alimentan los indicadores de `05_evaluation`. Las cajas para **entrenar** el detector NO salen de aquí: se pre-anotan y revisan por humanos en CVAT (`03_anotacion_cvat`), para evitar el entrenamiento circular.

**Salida:** `data/malaga/fases/detections.parquet` (fase `detections`), unida al resto por `(camera, image)`.

## 1. Configuración y modelo

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import load_config
from deteccion import frame_counts, predict_boxes
from pipeline import image_path, load_phase, save_phase

CFG = load_config()["deteccion"]
DEVICE = 0 if torch.cuda.is_available() else "cpu"

frames_df = load_phase("frames")
model = YOLO(CFG["modelo"])  # se descarga automáticamente la primera vez

print(f"Fotogramas: {len(frames_df)}")
print(f"Modelo: {CFG['modelo']} | conf={CFG['conf']} | imgsz={CFG['imgsz']} | dispositivo={DEVICE}")

## 2. Detección sobre todos los fotogramas

Para una prueba rápida puede limitarse con `MAX_FRAMES` (None = todos).

In [ ]:
MAX_FRAMES = None  # p.ej. 50 para una prueba rápida

work_df = frames_df if MAX_FRAMES is None else frames_df.head(MAX_FRAMES)

results = []
errors = []
for row in work_df.itertuples():
    img = cv2.imread(str(image_path(row.path)))
    if img is None:
        errors.append({"camera": row.camera, "image": row.image, "error": "no existe o no legible"})
        continue

    boxes = predict_boxes(model, img, conf=CFG["conf"], imgsz=CFG["imgsz"], device=DEVICE)
    results.append({
        "camera": row.camera,
        "image": row.image,
        **frame_counts(boxes, img.shape, min_group=CFG["min_personas_grupo"]),
    })

    if len(results) % 500 == 0:
        print(f"Procesadas {len(results)} imágenes...")

detect_df = pd.DataFrame(results)
print(f"\nImágenes procesadas: {len(detect_df)} | errores: {len(errors)}")
if errors:
    display(pd.DataFrame(errors).head())
display(detect_df.head(10))

## 3. Exportar la fase `detections`

In [ ]:
out = save_phase(detect_df, "detections")
print(f"Fase detections: {len(detect_df)} fotogramas -> {out}")

count_cols = [c for c in detect_df.columns if c.startswith("n_")]
print("\nTotales y fotogramas con presencia por clase:")
display(pd.DataFrame({
    "total": detect_df[count_cols].sum(),
    "fotogramas_con_presencia": (detect_df[count_cols] > 0).sum(),
    "media_por_fotograma": detect_df[count_cols].mean().round(2),
}))

## 4. Visualización de muestras (opcional)

Dibuja las detecciones sobre algunas imágenes para validar visualmente la pre-anotación.

In [ ]:
N_SAMPLES = 4

sample = frames_df.merge(detect_df[["camera", "image"]], on=["camera", "image"]).sample(
    min(N_SAMPLES, len(detect_df)), random_state=42
)
fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 4), squeeze=False)

for ax, row in zip(axes[0], sample.itertuples()):
    img = cv2.imread(str(image_path(row.path)))
    for name, (x1, y1, x2, y2) in predict_boxes(model, img, conf=CFG["conf"], imgsz=CFG["imgsz"], device=DEVICE):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
        cv2.putText(img, name, (int(x1), max(0, int(y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{row.camera} / {row.image}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()